
# Semana 2 — SQL avanzado

## Ejercicio 0.1: CTE básica — Estadísticas por zona

**Objetivo:** Utilizar una CTE para calcular estadísticas de los viajes de taxi de Nueva York

**Dataset:** `samples.nyctaxi.trips`

**Análisis:**
- Contar los viajes por zona de origen
- Calcular la tarifa promedio
- Calcular la distancia promedio
- Mostrar únicamente zonas con más de 100 viajes


In [0]:
WITH estadistica_por_zona AS(
    SELECT
        pickup_zip,
        count(*) AS total_viajes,
        AVG(fare_amount) AS promedio_tarifa,
        AVG(trip_distance) AS promedio_distancia
    FROM samples.nyctaxi.trips
    WHERE pickup_zip IS NOT NULL
    GROUP BY pickup_zip
    HAVING total_viajes > 100
    
)

SELECT *
FROM estadistica_por_zona;



# Ejercicio 0.2 — CTE múltiple: Análisis comparativo

## 1. Objetivo

Comparar la tarifa promedio de los viajes de taxi
de Nueva York según la hora de inicio con la tarifa
promedio general del conjunto de datos.

**Dataset:** `samples.nyctaxi.trips`

## 2. Metodología

Se utilizaron dos Common Table Expressions (CTEs).

### CTE 1: promedio_por_hora

Se extrajo la hora de inicio mediante HOUR() y
se agruparon los viajes utilizando GROUP BY.

Para cada hora se calcularon:

- La cantidad total de viajes mediante COUNT(*).
- La tarifa promedio mediante AVG(fare_amount).

### CTE 2: promedio_general

Se utilizó AVG(fare_amount) para calcular la
tarifa promedio de todos los viajes, sin agruparlos.

### Combinación de resultados

Se empleó CROSS JOIN para incorporar el promedio
general a cada uno de los 24 registros horarios.

Finalmente, se calculó la diferencia entre la
tarifa promedio por hora y la tarifa promedio general.

## 3. Resultados

Los resultados obtenidos fueron:

- **Tarifa promedio general:** $12.35.
- **Hora con mayor tarifa promedio:** 05:00.
- **Tarifa promedio máxima por hora:** $14.90.
- **Hora con menor tarifa promedio:** 19:00.
- **Tarifa promedio mínima por hora:** $11.36.
- **Hora con mayor cantidad de viajes:** 18:00,
  con 1,455 viajes.
- **Hora con menor cantidad de viajes:** 05:00,
  con 220 viajes.

## 4. Hallazgos

### Diferencias respecto al promedio general

A las 05:00 se registró una tarifa promedio de
$14.90, superior al promedio general en $2.55.

En cambio, a las 19:00 la tarifa promedio fue
$11.36, inferior al promedio general en $0.99.

Esto demuestra que el promedio de las tarifas
varía según la hora de inicio de los viajes.

### Relación entre cantidad de viajes y tarifas

La mayor cantidad de viajes se registró a las
18:00, con un total de 1,455 viajes y una
tarifa promedio de $11.77.

Por otro lado, a las 05:00 se registró la menor
cantidad de viajes, con 220 registros, pero
también la mayor tarifa promedio: $14.90.

Por tanto, la hora con más viajes no coincide
con la hora de mayor tarifa promedio.

Sin embargo, estos resultados no permiten
establecer una relación causal entre la cantidad
de viajes y el valor de las tarifas.

### Interpretación estadística

La tarifa promedio general fue de $12.35.

Este valor se calculó utilizando todos los
registros individuales, por lo que considera
automáticamente la cantidad de viajes de cada hora.

Al comparar las tarifas horarias con este promedio,
es posible identificar las horas que presentan
valores superiores o inferiores a la referencia.

## 5. Conclusión

El análisis permitió comparar las tarifas promedio
de los viajes según la hora del día utilizando
múltiples CTEs y CROSS JOIN.

Se observó que la tarifa promedio general fue de
$12.35, mientras que el mayor promedio horario
se registró a las 05:00, con $14.90, y el menor
a las 19:00, con $11.36.

Además, se identificó que la hora con mayor
cantidad de viajes (18:00) no presentó la
tarifa promedio más elevada.

Este resultado demuestra que la cantidad de viajes
por hora y la tarifa promedio son indicadores
diferentes y que no deben interpretarse como
si fueran equivalentes.

Desde el punto de vista estadístico, sería
conveniente profundizar en el análisis de las
distancias recorridas, los días de la semana
y los posibles valores atípicos antes de explicar
las causas de estas diferencias.

Desde el punto de vista de Data Engineering,
el ejercicio permitió aplicar CTEs para organizar
consultas complejas y CROSS JOIN para combinar
resultados agregados de diferentes niveles.

## 6. Conceptos aprendidos

- WITH: creación de múltiples CTEs.
- HOUR(): extracción de la hora.
- COUNT(*): conteo de registros.
- AVG(): cálculo de promedios.
- GROUP BY: agrupación de datos.
- CROSS JOIN: combinación de resultados.
- ROUND(): redondeo de valores.
- ORDER BY: ordenamiento de registros.
- AS: asignación de alias.

**Estado:** Ejercicio completado, ejecutado
y documentado con resultados reales.


In [0]:
WITH promedio_por_horas AS(
    SELECT
        
        hour(tpep_pickup_datetime) AS hora,
        COUNT(*) AS total_viajes,
        ROUND(AVG(fare_amount),2) AS tarifa_promedio
        
     FROM samples.nyctaxi.trips
     Group by hour(tpep_pickup_datetime) 
     
),

promedio_general AS (
    SELECT
        ROUND(AVG(fare_amount),2) AS tarifa_promedio_gerenal
    FROM samples.nyctaxi.trips

)

SELECT
    h.hora,
    h.total_viajes,
    h.tarifa_promedio,
    g.tarifa_promedio_gerenal,
    ROUND(h.tarifa_promedio-g.tarifa_promedio_gerenal,2) AS diferencia_tarifa
FROM promedio_por_horas AS h
CROSS JOIN promedio_general AS g

ORDER BY h.hora ASC;







# Ejercicio 0.3 — CTE para limpieza de datos

## 1. Objetivo

Aplicar una Common Table Expression (CTE)
para seleccionar viajes válidos y calcular
estadísticas descriptivas de sus distancias.

**Dataset:** `samples.nyctaxi.trips`

## 2. Metodología

Se creó una CTE llamada `viajes_validos` para
filtrar los registros que cumplen:

- `trip_distance > 0`
- `fare_amount > 0`

Posteriormente, sobre los registros filtrados
se calcularon las siguientes estadísticas:

- MIN(): distancia mínima.
- MAX(): distancia máxima.
- AVG(): distancia promedio.
- MEDIAN(): mediana de las distancias.

Este procedimiento permite separar la etapa
de filtrado de la etapa de análisis estadístico.

## 3. Resultados

Las estadísticas obtenidas fueron:

- **Distancia mínima:** 0.02 millas.
- **Distancia máxima:** 30.60 millas.
- **Distancia promedio:** 2.86 millas.
- **Mediana de distancia:** 1.70 millas.

## 4. Hallazgos

### Comparación entre promedio y mediana

La distancia promedio fue de 2.86 millas,
mientras que la mediana fue de 1.70 millas.

El promedio supera a la mediana en
aproximadamente 1.16 millas.

Esta diferencia sugiere una distribución
asimétrica hacia la derecha, posiblemente
influida por viajes de distancias elevadas.

### Análisis de valores extremos

La distancia mínima registrada fue de
0.02 millas y la máxima de 30.60 millas.

La distancia máxima es considerablemente
superior al promedio y a la mediana.

Por ello, sería recomendable analizar la
distribución mediante percentiles y diagramas
de caja para identificar posibles valores
atípicos.

Sin embargo, un viaje de distancia elevada
no necesariamente constituye un error.

### Importancia de la limpieza de datos

El uso de una CTE permitió excluir registros
con distancias o tarifas no positivas.

No obstante, estas condiciones representan
una validación inicial y no garantizan que
todos los viajes restantes sean correctos.

## 5. Conclusión

El ejercicio permitió aplicar una CTE como
herramienta para realizar un filtrado inicial
de los datos antes del análisis estadístico.

Después de filtrar los registros, se obtuvo
una distancia promedio de 2.86 millas y una
mediana de 1.70 millas.

La diferencia entre ambos indicadores sugiere
que la distribución de las distancias podría
presentar asimetría positiva.

Además, la distancia máxima de 30.60 millas
indica la conveniencia de investigar los
registros extremos para determinar si
corresponden a viajes válidos.

Desde una perspectiva de Data Engineering,
el ejercicio demuestra la importancia de
aplicar reglas de calidad de datos antes
de realizar transformaciones y análisis.

## 6. Conceptos aprendidos

- WITH: creación de una CTE.
- WHERE: filtrado de registros.
- MIN(): cálculo del valor mínimo.
- MAX(): cálculo del valor máximo.
- AVG(): cálculo del promedio.
- MEDIAN(): cálculo de la mediana.
- AS: asignación de alias.
- Limpieza y validación inicial de datos.
- Interpretación de estadísticas descriptivas.

**Estado:** Ejercicio completado, ejecutado
y documentado con resultados reales.


In [0]:
with viajes_validos AS (
    select
        trip_distance,
        fare_amount
    from samples.nyctaxi.trips
    where trip_distance > 0 and fare_amount >0
)

SELECT 
    MIN(trip_distance) AS distancia_minima,
    MAX(trip_distance) AS distancia_maxima,
    AVG(trip_distance) AS distancia_promedio,
    MEDIAN(trip_distance) AS mediana_de_distancia
FROM viajes_validos;



%md

# Ejercicio 0.4 — Window Function: ROW_NUMBER()

## Objetivo

Crear un ranking de los 20 viajes de taxi más caros
utilizando la función de ventana ROW_NUMBER().

**Dataset:** `samples.nyctaxi.trips`

## Metodología

Se utilizó ROW_NUMBER() junto con OVER() y
ORDER BY fare_amount DESC para asignar una
posición a cada viaje según su tarifa.

Posteriormente, se ordenaron los resultados
por ranking y se utilizó LIMIT 20 para mostrar
únicamente los 20 viajes más caros.

## Resultados

En los primeros registros obtenidos se observó:

- **Viaje más caro:** $275.
- **Distancia del viaje más caro:** 20.85 millas.
- **Segunda tarifa más alta:** $260.
- **Tercera tarifa más alta:** $188.

## Hallazgos

Se identificaron viajes con tarifas elevadas
que registran una distancia de cero millas.

Por ejemplo, los viajes con tarifas de $260
y $188 presentan una distancia registrada de 0.

Esto indica la necesidad de revisar la calidad
de los registros antes de utilizarlos para
realizar análisis estadísticos más profundos.

También se observaron tarifas repetidas.

Por ejemplo, dos viajes con tarifa de $105
recibieron las posiciones 6 y 7.

Esto demuestra que ROW_NUMBER() asigna números
consecutivos diferentes, incluso cuando existen
valores iguales en la columna de ordenamiento.

## Conclusión

El ejercicio permitió comprender cómo utilizar
ROW_NUMBER() para construir rankings sin
agrupar ni eliminar los registros individuales.

Se identificó que la tarifa más alta fue de
$275, correspondiente a un viaje de 20.85 millas.

Además, se encontraron viajes con tarifas
elevadas y distancias registradas de cero,
lo que sugiere la necesidad de aplicar
validaciones adicionales de calidad de datos.

Finalmente, se comprobó que ROW_NUMBER()
asigna posiciones únicas y consecutivas,
incluso cuando existen tarifas repetidas.

**Estado:** Ejercicio completado y documentado.


In [0]:
SELECT 
    tpep_pickup_datetime,
    trip_distance,
    fare_amount,
    ROW_NUMBER() OVER (
                ORDER BY fare_amount DESC  
    ) AS ranking 
FROM samples.nyctaxi.trips

ORDER BY ranking ASC 
LIMIT 20;





%md

# Ejercicio 0.5 — ROW_NUMBER() vs RANK() vs DENSE_RANK()

## 1. Objetivo

Comparar el funcionamiento de tres funciones de
ventana utilizadas para generar rankings en SQL,
analizando especialmente cómo manejan los empates.

**Dataset:** `samples.nyctaxi.trips`

## 2. Metodología

Se seleccionaron la fecha de inicio y la tarifa
de los viajes de taxi de Nueva York.

Posteriormente, se aplicaron tres funciones:

- ROW_NUMBER(): asigna números consecutivos únicos.
- RANK(): asigna el mismo puesto a los empates,
  dejando saltos en la numeración.
- DENSE_RANK(): asigna el mismo puesto a los
  empates, sin dejar saltos en la numeración.

Las tres funciones utilizaron:

`OVER (ORDER BY fare_amount DESC)`

Finalmente, se seleccionaron los 20 viajes
más caros mediante ORDER BY y LIMIT 20.

## 3. Resultados

La consulta devolvió 20 viajes ordenados
de mayor a menor tarifa.

Se identificaron las siguientes situaciones:

- Dos viajes con tarifa de $105.
- Tres viajes con tarifa de $95.
- Dos viajes con tarifa de $85.

Para los viajes de $95:

- ROW_NUMBER() asignó los números 8, 9 y 10.
- RANK() asignó el puesto 8 a los tres.
- DENSE_RANK() asignó el puesto 7 a los tres.

La siguiente tarifa, $94.50, recibió:

- ROW_NUMBER(): 11.
- RANK(): 11.
- DENSE_RANK(): 8.

## 4. Hallazgos

### Comportamiento de ROW_NUMBER()

Esta función asignó posiciones consecutivas,
independientemente de las tarifas repetidas.

Sin embargo, cuando existen empates y no se
especifica un criterio adicional de ordenamiento,
la asignación individual dentro del empate
no es necesariamente determinista.

### Comportamiento de RANK()

Los viajes con tarifas iguales recibieron
el mismo ranking.

Después de los tres viajes empatados en el
puesto 8, la siguiente tarifa recibió el
puesto 11, dejando saltos en la numeración.

### Comportamiento de DENSE_RANK()

Los viajes empatados también recibieron
el mismo ranking, pero sin generar saltos.

Después de los tres viajes con ranking 7,
la siguiente tarifa recibió el ranking 8.

### Comparación final

Dentro de los 20 viajes analizados se
identificaron 16 tarifas diferentes.

Por ello, el último registro presentó
ROW_NUMBER = 20, RANK = 20 y
DENSE_RANK = 16.

## 5. Conclusión

El ejercicio permitió comprender las diferencias
entre ROW_NUMBER(), RANK() y DENSE_RANK()
al construir clasificaciones sobre datos reales.

ROW_NUMBER() resulta útil cuando se necesita
asignar un identificador secuencial a cada fila.

RANK() resulta apropiado cuando se desean
respetar los empates y conservar las posiciones
que ocuparían los registros empatados.

DENSE_RANK() permite clasificar valores
distintos sin dejar posiciones vacantes.

La elección de una función depende de las
necesidades del análisis y del tratamiento
que se quiera dar a los empates.

Desde una perspectiva de Data Engineering,
estas funciones facilitan el análisis y la
clasificación de grandes conjuntos de datos
sin necesidad de agrupar las filas originales.

## 6. Conceptos aprendidos

- ROW_NUMBER()
- RANK()
- DENSE_RANK()
- OVER()
- ORDER BY DESC
- LIMIT
- Manejo de empates en rankings

**Estado:** Ejercicio completado, ejecutado
y documentado con resultados reales.


In [0]:
SELECT 
    tpep_pickup_datetime,
    fare_amount,
    

    ROW_NUMBER() OVER (
                ORDER BY fare_amount DESC   
    ) AS ranking_row_numbe, 

    RANK() OVER(
        ORDER BY fare_amount DESC   
    ) AS ranking_rank, 

    DENSE_RANK() OVER(
        ORDER BY fare_amount DESC
    ) AS ranking_dense_rank
FROM samples.nyctaxi.trips

ORDER BY fare_amount DESC 
LIMIT 20;


# Ejercicio 0.6 — Comparación con el promedio general

## 1. Objetivo

Utilizar la función de ventana AVG() OVER() para
comparar la tarifa de cada viaje de taxi con la
tarifa promedio general del conjunto de datos.

Se busca calcular tanto la diferencia absoluta
como la diferencia porcentual de cada viaje
respecto al promedio general.

**Dataset:** `samples.nyctaxi.trips`

## 2. Metodología

Se utilizó la función AVG(fare_amount) OVER()
para calcular la tarifa promedio general de
todos los registros sin perder la información
individual de cada viaje.

Posteriormente, se calcularon dos indicadores:

### Diferencia absoluta

Se restó la tarifa promedio general a la tarifa
individual de cada viaje.

**Fórmula:**

Diferencia absoluta = Tarifa individual - Promedio general

Una diferencia positiva indica que el viaje
tiene una tarifa superior al promedio general,
mientras que una diferencia negativa indica
que su tarifa es inferior.

### Diferencia porcentual

Se calculó la diferencia relativa utilizando
el promedio general como referencia.

**Fórmula:**

Diferencia porcentual =
((Tarifa individual - Promedio general)
/ Promedio general) * 100

Este indicador permite expresar cuánto se aleja
cada tarifa del promedio general en términos
porcentuales.

Finalmente, se utilizó ROUND() para redondear
los resultados y LIMIT 10 para visualizar
diez registros del conjunto de datos.

## 3. Resultados

La tarifa promedio general obtenida fue de
**$12.35**.

En los diez registros visualizados se encontraron:

- **5 viajes** con tarifas superiores al promedio.
- **5 viajes** con tarifas inferiores al promedio.

Entre los resultados más relevantes:

- Un viaje de **$31.00** presentó una diferencia
  positiva de **$18.65**, equivalente
  aproximadamente al **151 %**.

- Un viaje de **$28.50** presentó una diferencia
  positiva de **$16.15**, equivalente
  aproximadamente al **131 %**.

- Un viaje de **$6.50** presentó una diferencia
  negativa de **$5.85**, equivalente
  aproximadamente al **-47 %**.

- Un viaje de **$11.50** presentó una diferencia
  negativa de **$0.85**, equivalente
  aproximadamente al **-7 %**.

## 4. Hallazgos

### Comparación de tarifas individuales

Se identificaron diferencias entre las tarifas
de los viajes individuales y el promedio general
del conjunto de datos.

Los viajes con tarifas de $31.00 y $28.50
presentaron las mayores diferencias positivas
entre los diez registros visualizados.

Por otro lado, el viaje con tarifa de $6.50
registró la mayor diferencia negativa.

### Interpretación porcentual

Las diferencias porcentuales permiten comparar
la magnitud relativa de las tarifas individuales.

Por ejemplo, una diferencia positiva del 151 %
indica que la tarifa individual supera al promedio
general en aproximadamente 1.51 veces el valor
de dicho promedio.

En cambio, una diferencia negativa del -47 %
indica que la tarifa individual está aproximadamente
un 47 % por debajo del promedio general.

### Importancia de las funciones de ventana

A diferencia de una agregación tradicional,
AVG() OVER() permitió conservar cada registro
individual y añadir una columna con el promedio
general.

Esto facilitó realizar comparaciones sin
necesidad de agrupar los datos mediante GROUP BY
ni combinar resultados con CROSS JOIN.

### Consideraciones estadísticas

Los resultados visualizados corresponden
únicamente a diez registros seleccionados
mediante LIMIT 10.

Por tanto, no es posible concluir que el 50 %
de todos los viajes se encuentra por encima
del promedio general.

Para realizar afirmaciones sobre la distribución
completa de las tarifas sería necesario analizar
todos los registros del conjunto de datos.

Además, el promedio puede verse influido por
tarifas extremadamente altas o bajas, por lo
que sería conveniente complementar el análisis
con la mediana y otras medidas descriptivas.

## 5. Conclusión

El ejercicio permitió aplicar la función de
ventana AVG() OVER() para comparar las tarifas
individuales de los viajes con el promedio
general del conjunto de datos.

Se obtuvo una tarifa promedio general de $12.35,
que sirvió como referencia para calcular las
diferencias absolutas y porcentuales.

En los diez registros visualizados se identificaron
tarifas superiores e inferiores al promedio,
destacándose un viaje de $31.00 que presentó
una diferencia positiva aproximada del 151 %.

Desde una perspectiva estadística, el análisis
permitió comprender cómo interpretar las
desviaciones individuales respecto a una medida
de tendencia central, diferenciando entre
comparaciones absolutas y relativas.

Desde la perspectiva de Data Engineering,
se comprobó que las funciones de ventana
permiten incorporar estadísticas generales
a cada registro sin perder el detalle
de los datos originales.

Finalmente, se recomienda complementar este
análisis con otras medidas descriptivas y
explorar posibles valores atípicos para
comprender mejor la distribución de las tarifas.

## 6. Conceptos aprendidos

- **AVG():** cálculo del promedio aritmético.
- **OVER():** aplicación de una función de ventana
  sobre los registros de la consulta.
- **AVG() OVER():** cálculo del promedio general
  sin eliminar los registros individuales.
- **ROUND():** redondeo de resultados numéricos.
- **LIMIT:** limitación del número de filas
  mostradas en el resultado.
- **AS:** asignación de alias a las columnas.
- **Diferencia absoluta:** comparación monetaria
  entre una tarifa y el promedio general.
- **Diferencia porcentual:** comparación relativa
  de una tarifa respecto al promedio general.

**Estado:** Ejercicio completado, ejecutado
y documentado con resultados reales.


In [0]:
SELECT
    tpep_pickup_datetime,
    trip_distance,
    fare_amount,

    ROUND(AVG(fare_amount) OVER (),2
         
    )AS promedio_general,

    ROUND(
        fare_amount-AVG(fare_amount) OVER(), 2

    ) AS diferencia_tarifa,

    ROUND(
        (
            ((fare_amount-AVG(fare_amount) OVER())/AVG(fare_amount) OVER())*100 )
        
        )AS diferencia_porcentual 
    

FROM samples.nyctaxi.trips
LIMIT 10; 



# Ejercicio 0.7 — Promedio por partición con PARTITION BY

## 1. Objetivo

Calcular la tarifa promedio de los viajes de taxi
según su zona de origen utilizando la función
de ventana AVG() OVER (PARTITION BY pickup_zip).

El propósito es conservar la información individual
de cada viaje y añadir el promedio correspondiente
a su zona de origen.

**Dataset:** `samples.nyctaxi.trips`

## 2. Metodología

Se seleccionaron las siguientes columnas:

- tpep_pickup_datetime: fecha de inicio del viaje.
- pickup_zip: zona de origen.
- fare_amount: tarifa individual.

Se utilizó la expresión:

`AVG(fare_amount) OVER (PARTITION BY pickup_zip)`

Esta función calcula la tarifa promedio de todos
los viajes que pertenecen a una misma zona,
sin eliminar los registros individuales.

Posteriormente, se aplicó ROUND() para redondear
el promedio de cada zona a dos decimales.

Se utilizó ORDER BY pickup_zip para ordenar
los registros según su zona de origen.

Finalmente, LIMIT 20 permitió visualizar
únicamente veinte registros del resultado.

## 3. Resultados

La consulta devolvió veinte registros y
cuatro columnas:

- fecha_viaje
- zona_inicio
- tarifa
- promedio_zona

Entre los resultados observados:

- La zona 7002 presentó un promedio de $10.50.
- La zona 10001 presentó un promedio de $10.62.
- La zona 7974 presentó un promedio de $188.00.
- La zona 8876 presentó un promedio de $260.00.

## 4. Hallazgos

### Promedios por zona

Se observó que los viajes pertenecientes a
una misma zona comparten el mismo promedio,
aunque sus tarifas individuales sean diferentes.

Por ejemplo, en la zona 7002 se visualizaron
viajes con tarifas de $8.50 y $12.50.
Ambos registros mostraron un promedio de $10.50.

### Análisis de la zona 10001

En la zona 10001 se visualizaron viajes con
tarifas diferentes, incluyendo valores de
$4.00, $9.50, $10.50 y $40.00.

Sin embargo, todos mostraron una tarifa
promedio de zona de $10.62.

Esto evidencia que cada registro conserva
su tarifa individual y recibe una estadística
calculada a partir de su zona de origen.

### Interpretación estadística

El promedio de cada zona se calcula utilizando
los registros que pertenecen a esa zona dentro
del conjunto de datos consultado.

Por tanto, LIMIT 20 únicamente restringe
la cantidad de registros visualizados y no
la cantidad de viajes utilizados para calcular
los promedios.

Las diferencias entre las tarifas individuales
y sus promedios de zona permiten observar
variaciones dentro de una misma categoría.

Sin embargo, para evaluar la dispersión de
las tarifas sería necesario calcular medidas
adicionales, como la desviación estándar
o los percentiles.

## 5. Conclusión

El ejercicio permitió aplicar la función
AVG() OVER (PARTITION BY pickup_zip) para
calcular tarifas promedio según la zona
de origen de los viajes.

Se observó que los registros pertenecientes
a una misma zona presentan un promedio
compartido, aunque sus tarifas individuales
sean diferentes.

Por ejemplo, la zona 10001 presentó un
promedio de $10.62 en los registros
visualizados, incluso cuando algunas
tarifas individuales fueron superiores
o inferiores a dicho valor.

Desde una perspectiva estadística, este
procedimiento permite incorporar medidas
de tendencia central específicas de cada
categoría y facilitar comparaciones entre
registros de una misma zona.

Desde una perspectiva de Data Engineering,
el ejercicio demuestra la utilidad de las
funciones de ventana para enriquecer los
registros individuales con información
estadística de su grupo.

## 6. Conceptos aprendidos

- AVG(): cálculo de la tarifa promedio.
- OVER(): aplicación de una función de ventana.
- PARTITION BY: definición de particiones
  según la zona de origen.
- ROUND(): redondeo de valores numéricos.
- ORDER BY: ordenamiento de registros.
- LIMIT: limitación de filas visualizadas.
- AS: asignación de alias a las columnas.

**Estado:** Ejercicio completado, ejecutado
y documentado con resultados reales.


In [0]:
SELECT 
    tpep_pickup_datetime AS fecha_viaje,
    pickup_zip AS zona_inicio,
    fare_amount AS tarifa,

    ROUND(
        AVG(fare_amount) OVER (
            PARTITION BY pickup_zip
        ), 2
    ) AS promedio_zona

FROM samples.nyctaxi.trips
ORDER BY pickup_zip 
LIMIT 20;
    



%md

# Ejercicio 0.8 — Comparación con el registro anterior mediante LAG()

## 1. Objetivo

Utilizar la función de ventana LAG() para
consultar la tarifa del viaje anterior según
el orden cronológico de los registros.

El propósito es conservar la información de
cada viaje e incorporar la tarifa correspondiente
al registro precedente.

**Dataset:** `samples.nyctaxi.trips`

## 2. Metodología

Se seleccionaron dos columnas principales:

- tpep_pickup_datetime: fecha de inicio del viaje.
- fare_amount: tarifa individual del viaje.

Posteriormente, se utilizó la expresión:

`LAG(fare_amount) OVER (ORDER BY tpep_pickup_datetime)`

Esta función permite recuperar la tarifa
del registro anterior considerando el orden
de las fechas de inicio de los viajes.

Se utilizó ORDER BY para presentar los
resultados cronológicamente y LIMIT 20
para visualizar veinte registros.

## 3. Resultados

La consulta devolvió 20 registros con
tres columnas:

- fecha_viaje
- tarifa_actual
- tarifa_anterior

Entre los resultados obtenidos:

- El primer viaje registrado tuvo una tarifa
  de $5.00 y una tarifa anterior NULL.

- El segundo viaje presentó una tarifa
  de $24.50 y una tarifa anterior de $5.00.

- El tercer viaje presentó una tarifa
  de $5.00 y una tarifa anterior de $24.50.

- El cuarto viaje presentó una tarifa
  de $10.00 y una tarifa anterior de $5.00.

## 4. Hallazgos

### Recuperación del registro anterior

Se comprobó que LAG() permite acceder a
la tarifa de un viaje anterior sin modificar
ni eliminar los registros individuales.

Cada fila conserva su tarifa original e
incorpora una columna adicional con la
tarifa del registro precedente.

### Interpretación de valores NULL

El primer registro mostró NULL en la columna
tarifa_anterior porque no existe un viaje
anterior dentro del orden cronológico.

Este comportamiento es esperado cuando
se utiliza LAG() sin especificar un valor
alternativo para los registros inexistentes.

### Variaciones entre viajes consecutivos

Se observaron diferencias entre las tarifas
de viajes consecutivos.

Por ejemplo, el segundo viaje tuvo una
tarifa de $24.50 frente a los $5.00
del registro anterior.

Posteriormente, el tercer viaje presentó
una tarifa de $5.00 frente a los $24.50
del viaje precedente.

Estas variaciones muestran diferencias
entre registros, pero no son suficientes
para establecer una tendencia general.

### Consideraciones del orden cronológico

La función LAG() utiliza la fecha de inicio
para determinar qué registro es anterior.

Si dos viajes tienen exactamente la misma
fecha y hora, el orden entre ellos puede
ser ambiguo porque no se estableció un
criterio adicional de desempate.

## 5. Conclusión

El ejercicio permitió comprender cómo
utilizar la función de ventana LAG()
para recuperar información de un registro
anterior dentro de una secuencia ordenada.

Se comprobó que el primer registro presenta
NULL como tarifa anterior y que los registros
posteriores recuperan las tarifas precedentes.

Los resultados muestran que LAG() facilita
la comparación de observaciones consecutivas,
sin necesidad de eliminar o agrupar registros.

Desde una perspectiva estadística, esta técnica
es útil como punto de partida para estudiar
variaciones entre observaciones ordenadas.

Desde una perspectiva de Data Engineering,
LAG() permite incorporar información histórica
a cada registro y facilita la construcción
de análisis sobre secuencias temporales.

Es importante considerar que los viajes
consecutivos del dataset no necesariamente
pertenecen al mismo vehículo o conductor.

## 6. Conceptos aprendidos

- LAG(): recuperación del valor de una fila anterior.
- OVER(): definición de la función de ventana.
- ORDER BY: ordenamiento cronológico de registros.
- LIMIT: limitación de los registros visualizados.
- AS: asignación de alias a las columnas.
- NULL: ausencia de un valor anterior disponible.

**Estado:** Ejercicio completado, ejecutado
y documentado con resultados reales.


In [0]:
SELECT
    tpep_pickup_datetime AS fecha_viaje,
    fare_amount AS tarifa_actual ,

    LAG(fare_amount) OVER (
        ORDER BY tpep_pickup_datetime
    ) AS tarifa_anterior
FROM samples.nyctaxi.trips

ORDER BY tpep_pickup_datetime ASC
LIMIT 20;    


%md

# Ejercicio 0.9 — Suma acumulada con SUM() OVER()

## 1. Objetivo

Utilizar la función de ventana SUM() OVER()
para calcular el acumulado de las tarifas
de los viajes de taxi ordenados cronológicamente.

El propósito es observar cómo aumenta el
total acumulado a medida que se incorporan
nuevos registros.

**Dataset:** `samples.nyctaxi.trips`

## 2. Metodología

Se seleccionaron las siguientes columnas:

- tpep_pickup_datetime: fecha de inicio del viaje.
- fare_amount: tarifa individual del viaje.

Posteriormente, se utilizó la expresión:

`SUM(fare_amount) OVER (
    ORDER BY tpep_pickup_datetime
    ROWS BETWEEN UNBOUNDED PRECEDING
             AND CURRENT ROW
)`

Esta función calcula una suma acumulada
desde el primer registro hasta la fila actual.

Se utilizó ORDER BY para presentar los
registros cronológicamente y LIMIT 20
para visualizar los primeros veinte viajes.

## 3. Resultados

La consulta devolvió veinte registros
y tres columnas:

- fecha_viaje
- tarifa
- tarifa_acumulada

Entre los resultados obtenidos:

- Primer viaje: tarifa de $5.00 y
  acumulado de $5.00.

- Segundo viaje: tarifa de $24.50 y
  acumulado de $29.50.

- Tercer viaje: tarifa de $5.00 y
  acumulado de $34.50.

- Décimo viaje: tarifa de $10.00 y
  acumulado de $96.50.

- Vigésimo viaje: tarifa de $27.00 y
  acumulado de $231.50.

## 4. Hallazgos

### Comportamiento del acumulado

Se observó que el valor de tarifa_acumulada
aumenta progresivamente conforme se
incorporan los viajes.

En los veinte registros visualizados,
todas las tarifas fueron positivas,
por lo que el acumulado no disminuyó.

### Interpretación del total acumulado

El primer registro presentó una tarifa
acumulada de $5.00.

Al incorporar el segundo viaje, con una
tarifa de $24.50, el acumulado aumentó
hasta $29.50.

Después de los primeros veinte viajes,
el acumulado alcanzó $231.50.

Este valor representa la suma de las tarifas
de los primeros veinte viajes del orden
cronológico utilizado en la consulta.

### Importancia de ROWS BETWEEN

La instrucción ROWS BETWEEN UNBOUNDED PRECEDING
AND CURRENT ROW permitió especificar
que cada cálculo incluye desde la primera
fila hasta la fila actual.

Por tanto, el acumulado incorpora las tarifas
de los registros precedentes y del viaje
correspondiente a cada fila.

### Consideraciones del orden cronológico

El acumulado depende del orden establecido
mediante tpep_pickup_datetime.

Si existen registros con fechas y horas
idénticas, el orden entre ellos puede
ser ambiguo al no existir un criterio
adicional de desempate.

Por ello, para aplicaciones que requieren
resultados reproducibles sería recomendable
establecer un orden completamente definido.

## 5. Conclusión

El ejercicio permitió aplicar SUM() OVER()
para calcular sumas acumuladas sin perder
la información individual de cada viaje.

Se comprobó que el acumulado incorpora
progresivamente las tarifas conforme
avanzamos en el orden cronológico.

Los primeros veinte viajes presentaron
un total acumulado de $231.50, mientras
que el primer registro comenzó con $5.00.

Desde una perspectiva estadística,
este procedimiento facilita analizar
la evolución de una magnitud acumulada
a lo largo de una secuencia ordenada.

Desde la perspectiva de Data Engineering,
las funciones de ventana permiten generar
indicadores acumulativos conservando
el detalle de los registros originales.

## 6. Conceptos aprendidos

- SUM(): cálculo de sumas.
- OVER(): aplicación de funciones de ventana.
- ORDER BY: definición del orden cronológico.
- ROWS BETWEEN: delimitación de la ventana.
- UNBOUNDED PRECEDING: inicio de la ventana
  desde la primera fila.
- CURRENT ROW: inclusión de la fila actual.
- LIMIT: limitación de registros visualizados.
- AS: asignación de alias a las columnas.

**Estado:** Ejercicio completado, ejecutado
y documentado con resultados reales.


In [0]:
SELECT
    tpep_pickup_datetime AS fecha_viaje,
    fare_amount AS tarifa,

    SUM(fare_amount) OVER (
        ORDER BY tpep_pickup_datetime
        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
    ) AS tarifa_acumulada 

FROM samples.nyctaxi.trips
ORDER BY tpep_pickup_datetime ASC
LIMIT 20;
    


%md

# Ejercicio 0.10 — Combinación de CTEs y Window Functions

## 1. Objetivo

Combinar Common Table Expressions (CTEs) y
funciones de ventana para identificar las
10 zonas de origen con mayor tarifa promedio.

El análisis considera únicamente viajes válidos,
con distancia y tarifa mayores que cero.

**Dataset:** `samples.nyctaxi.trips`

## 2. Metodología

### Primera CTE: viajes_validos

Se creó una CTE llamada `viajes_validos` para
seleccionar los registros que cumplen:

- trip_distance > 0
- fare_amount > 0
- pickup_zip IS NOT NULL

Estos filtros permiten excluir distancias y
tarifas no positivas, así como zonas sin
identificación.

### Segunda CTE: estadisticas_por_zona

Se utilizó la CTE `viajes_validos` como fuente
para calcular estadísticas por zona de origen.

Mediante GROUP BY pickup_zip se calcularon:

- AVG(fare_amount): tarifa promedio.
- MAX(fare_amount): tarifa máxima.
- MIN(fare_amount): tarifa mínima.

### Construcción del ranking

Se utilizó ROW_NUMBER() OVER() para asignar
una posición a cada zona según su tarifa
promedio, ordenada de mayor a menor.

Finalmente, se aplicó LIMIT 10 para mostrar
las diez primeras zonas del ranking.

## 3. Resultados

La consulta devolvió 10 registros y 5 columnas:

- ranking
- zona
- tarifa_promedio
- tarifa_maxima
- tarifa_minima

Entre los principales resultados:

- Zona 7311: tarifa promedio de $60.00.
- Zona 11430: tarifa promedio de $52.00.
- Zona 11368: tarifa promedio de $52.00.
- Zona 11420: tarifa promedio de $52.00.
- Zona 11435: tarifa promedio de $52.00.
- Zona 11422: tarifa promedio de $45.23.
- Zona 11436: tarifa promedio de $42.32.
- Zona 7030: tarifa promedio de $40.00.
- Zona 11424: tarifa promedio de $34.25.
- Zona 11235: tarifa promedio de $33.25.

## 4. Hallazgos

### Zona con mayor tarifa promedio

La zona 7311 ocupó la primera posición,
con una tarifa promedio de $60.00.

Su tarifa mínima y máxima también fueron
de $60.00 dentro de los registros considerados.

Sin embargo, esta coincidencia no permite
determinar cuántos viajes existen en la zona.

### Zonas con promedios similares

Las zonas 11430, 11368, 11420 y 11435
presentaron tarifas promedio mostradas
de $52.00.

Cada zona recibió una posición diferente,
debido al uso de ROW_NUMBER().

### Variación de las tarifas

La zona 11422 presentó una tarifa mínima
de $3.00 y una máxima de $82.50.

La diferencia entre ambos extremos fue
de $79.50.

Por otra parte, la zona 11436 presentó
tarifas entre $5.00 y $75.00.

Estos resultados muestran que las tarifas
pueden variar considerablemente dentro
de determinadas zonas.

### Consideraciones estadísticas

El ranking fue construido después de
aplicar filtros para seleccionar viajes
con distancias y tarifas positivas.

Por tanto, los promedios representan
únicamente los registros que cumplen
estas condiciones.

Además, sería conveniente conocer el
número de viajes por zona para evaluar
la cantidad de observaciones utilizadas
en el cálculo de cada promedio.

## 5. Conclusión

El ejercicio permitió integrar CTEs y
funciones de ventana para construir un
análisis de tarifas por zona de origen.

La primera CTE permitió filtrar los viajes
según condiciones de validez.

La segunda CTE permitió calcular la tarifa
promedio, máxima y mínima de cada zona.

Posteriormente, ROW_NUMBER() permitió
clasificar las zonas según su tarifa
promedio e identificar las diez primeras.

La zona 7311 presentó el promedio más
elevado, con $60.00.

También se identificaron diferencias
importantes entre las tarifas mínimas
y máximas de algunas zonas, especialmente
en las zonas 11422 y 11436.

Desde una perspectiva estadística, el
análisis permitió comparar medidas
descriptivas entre distintas zonas.

Desde la perspectiva de Data Engineering,
el ejercicio demuestra cómo organizar
transformaciones sucesivas mediante CTEs
y aplicar funciones de ventana para
construir rankings sobre los resultados.

Como mejora posterior, sería conveniente
incorporar el número de viajes por zona
para interpretar los promedios considerando
el tamaño de cada grupo.

## 6. Conceptos aprendidos

- WITH: definición de múltiples CTEs.
- WHERE: filtrado de registros.
- AND: combinación de condiciones.
- IS NOT NULL: exclusión de zonas nulas.
- AVG(): cálculo de tarifas promedio.
- MAX(): identificación de tarifas máximas.
- MIN(): identificación de tarifas mínimas.
- GROUP BY: agrupación de viajes por zona.
- ROW_NUMBER(): asignación de posiciones.
- OVER(): aplicación de funciones de ventana.
- ORDER BY: ordenamiento de resultados.
- ROUND(): presentación de promedios redondeados.
- LIMIT: selección de los diez primeros registros.
- AS: asignación de alias.

**Estado:** Ejercicio ejecutado y documentado
con resultados reales.


In [0]:
WITH viajes_validos AS (
    SELECT
        pickup_zip, 
        fare_amount,
        trip_distance
    FROM
        samples.nyctaxi.trips
    WHERE
        trip_distance > 0 AND
        fare_amount > 0  AND 
        pickup_zip IS NOT NULL

),

estadistica_por_zona AS(
    SELECT
        pickup_zip AS zona, 
        ROUND(AVG(fare_amount),2) AS tarifa_promedio,
        MAX(fare_amount) AS tarifa_maxima,
        MIN(fare_amount) AS tarifa_minima
    FROM viajes_validos
    GROUP BY pickup_zip
)

SELECT
    row_number() OVER(ORDER BY tarifa_promedio DESC
    ) AS ranking,
    zona, 
    ROUND(tarifa_promedio,2) AS tarifa_promedio,
    tarifa_maxima,
    tarifa_minima
FROM estadistica_por_zona
ORDER BY ranking ASC
LIMIT 10;